# CSIEM 1990s ocean assessment — obs vs BC vs TUFLOW-FV

Synoptic diagnosis of model & boundary-condition bias against the SMCWS outer-ring CTD casts (1991–1994).
Three sources per cast, **surface (top-2 m)** and **bottom (bottom-2 m)**, tagged to 5 latitude-band subregions (N/NW/W/SW/S):

- **OBS** — SMCWS field casts (reusing `compare_to_roms.py` binary readers)
- **BC** — the model boundary forcing actually used: ROMS climatology (1991/1992) + HYCOM (1993/1994), from `environment_repo`
- **MODEL** — TUFLOW-FV `TEMP`/`SAL`, where a run window covers the cast date (1991-Aug, 1992 Feb–May; 1993/94 model output is effectively absent)

**Verified headline (independent recompute + adversarial audit):** the model is **too salty at the surface and bottom by ~+0.5–0.7 psu in every subregion** (robust under all stress tests; the BC is slightly *fresh*, so the salt excess is model-internal). Temperature is warm-biased except the South. See `FINDINGS.md`.


In [ ]:
# --- setup: import the assessment modules ---
import os, sys, importlib
sys.path.insert(0, r'G:/CSIEM/1.8.0/csiem-marvl/custom_py/climatology_assessment')
import ocean_assessment_core as core
import ocean_assessment_plots as P
from IPython.display import Image, display

# Re-run the full 451-cast multi-source extraction (~80 s)? Default: use the existing CSV.
REEXTRACT = False
if REEXTRACT:
    df = core.run()
    df.to_csv(os.path.join(core.OUT_DIR, 'extended_assessment.csv'), index=False)
    importlib.reload(P)   # reload so plots pick up the fresh CSV (+ QC)
print('casts:', len(P.df), '| model-covered:', int(P.df.model_surfS.notna().sum()))

In [ ]:
# --- 1. Scatter: recreate obs-vs-BC + add the model (surface & bottom, T & S, by subregion) ---
P.scatter_fig('surf'); P.scatter_fig('bot')
display(Image(os.path.join(P.DIR, 'scatter_surf.png')))
display(Image(os.path.join(P.DIR, 'scatter_bot.png')))

In [ ]:
# --- 2. Subregion bias summary (BC vs MODEL; positive = too warm/salty) ---
summ = P.bias_summary()
display(Image(os.path.join(P.DIR, 'subregion_bias_summary.png')))
print(summ.pivot_table(index='subregion', columns=['source','level','var'], values='bias').round(3))

In [ ]:
# --- 3. Seasonal alignment per subregion: ROMS climatology curve + obs (by year) + TUFLOW-FV ---
for sr in P.SUB:
    P.seasonal_fig(sr)
for sr in P.SUB:
    display(Image(os.path.join(P.DIR, f'seasonal_{sr}.png')))

In [ ]:
# --- 4. Per-cast profile comparisons: obs + BC (ROMS-clim/HYCOM) + TUFLOW-FV (where covered) ---
# Redo of the original compare_to_roms per-cast profiles, with the model profile overlaid.
import ocean_assessment_profiles as PR
PREVIEW_ONLY = True   # True: render a couple of model-covered casts inline; False: write all ~404 to profiles/{year}/
if PREVIEW_ONLY:
    import csv, pandas as pd
    rows = list(csv.DictReader(open(PR.INV)))
    covered = [r for r in rows if any(pd.Timestamp(a) <= pd.Timestamp(r['date']) <= pd.Timestamp(b) for a,b,_ in PR.core.FV_RUNS)]
    for r in covered[:2]:
        out, had = PR.profile_fig(r)
        display(Image(out))
else:
    PR.run()   # all casts -> climatology_assessment/profiles/{year}/{date}_{station}_{time}.png

In [ ]:
# --- 5. Background BC envelope: coastal (E, bias-corrected) vs ocean (W) climatology ---
# The ROMS climatology is bias-corrected ONLY inside the seaward-arc polygons (tuned for
# 2013-2023): coastal/east cells swing salty-in-summer / fresh-in-winter; ocean/west cells
# are steadier. The model is forced by these corrected arc values, NOT the uncorrected
# interior cells the scatter 'BC' column sampled. These curves show the BC the model sees.
import ocean_assessment_bc_envelope as BCE
for sr in BCE.SUB:
    BCE.fig_sub(sr)
for sr in BCE.SUB:
    display(Image(os.path.join(BCE.DIR, f'seasonal_envelope_{sr}.png')))

In [ ]:
# --- 6. Date-axis BC time-series 1991-1994 (year-on-year ROMS-clim vs HYCOM) ---
# ROMS climatology tiled per year + HYCOM at actual dates, both at coastal/ocean polygons,
# with obs + TUFLOW-FV at true sampling dates. Shows the salt bias is tied to the ROMS
# coastal correction (1992) and absent under HYCOM forcing (1994).
import ocean_assessment_bc_timeseries as BCT
for sr in BCT.SUB:
    BCT.fig_sub(sr)
for sr in BCT.SUB:
    display(Image(os.path.join(BCT.DIR, f'bc_timeseries_{sr}.png')))